# GSM Thermodynamic Box 2 - Interface-based Interactive Widget Demo

This notebook demonstrates the interface-based implementation of the GSM Thermodynamic Box with an interactive widget.

The GSMThermodynBox2 uses interface-based state function implementations (GSMStateFnIfc) instead of raw expressions, making the framework more modular and extensible.

In [1]:
import sys
import os
from pathlib import Path

# Add the parent directory to the path so we can import the modules
current_dir = Path().resolve()
parent_dir = current_dir.parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

print(f"Current directory: {current_dir}")
print(f"Parent directory: {parent_dir}")
print(f"Python path includes parent: {str(parent_dir) in sys.path}")

Current directory: /home/rch/Coding/bmcs_matmod/notebooks
Parent directory: /home/rch/Coding/bmcs_matmod
Python path includes parent: True


In [ ]:
# Check availability of required modules
try:
    from bmcs_matmod.gsm_lagrange.core2 import gsm_vars
    gsm_vars_available = True
    print("✓ gsm_vars module is available")
except ImportError as e:
    gsm_vars_available = False
    print(f"✗ gsm_vars not available: {e}")

try:
    from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn import GSMStateFn, StateFunction
    gsm_state_fn_available = True
    print("✓ GSMStateFn is available")
except ImportError as e:
    gsm_state_fn_available = False
    print(f"✗ GSMStateFn not available: {e}")

try:
    from bmcs_matmod.gsm_lagrange.core2.gsm_thermodyn_box import GSMThermodynBox
    gsm_thermodyn_box2_available = True
    print("✓ GSMThermodynBox2 is available")
except ImportError as e:
    gsm_thermodyn_box2_available = False
    print(f"✗ GSMThermodynBox2 not available: {e}")

try:
    from bmcs_matmod.gsm_lagrange.core2.gsm_thermodyn_box_widget import GSMThermodynBoxWidget, create_interactive_widget
    gsm_widget_available = True
    print("✓ GSMThermodynBox2Widget is available")
except ImportError as e:
    gsm_widget_available = False
    print(f"✗ GSMThermodynBox2Widget not available: {e}")

✓ gsm_vars module is available
✓ GSMStateFn is available
✓ GSMThermodynBox2 is available
✓ GSMThermodynBox2Widget is available


In [3]:
# Import required libraries
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, Math, clear_output
from typing import Dict, List, Optional

print("Basic imports successful")

Basic imports successful


## Create Variables and Initial State Function

We'll create a simple thermodynamic state function to start with. Let's begin with the Helmholtz free energy F(T, ε, Ɛ) as our initial state function.

In [5]:
# Create symbolic variables
T, S = sp.symbols('T S', real=True)  # Temperature and entropy
eps, sig = sp.symbols('epsilon sigma', real=True)  # Strain and stress
Eps, Sig = sp.symbols('Epsilon Sigma', real=True)  # Internal strain and stress

# Material parameters
c_T, E, H = sp.symbols('c_T E H', positive=True)  # Heat capacity, Young's modulus, hardening

print("Variables created:")
print(f"Thermal: T = {T}, S = {S}")
print(f"Mechanical: ε = {eps}, σ = {sig}")
print(f"Internal: Ɛ = {Eps}, 𝒮 = {Sig}")
print(f"Parameters: c_T = {c_T}, E = {E}, H = {H}")

Variables created:
Thermal: T = T, S = S
Mechanical: ε = epsilon, σ = sigma
Internal: Ɛ = Epsilon, 𝒮 = Sigma
Parameters: c_T = c_T, E = E, H = H


In [6]:
# Create a simple Helmholtz free energy expression F(T, ε, Ɛ)
# F = thermal term + mechanical term + internal term + coupling terms
F_expr = (
    -c_T * T**2 / 2 +  # Thermal contribution (quadratic in T)
    E * eps**2 / 2 +   # Mechanical elastic energy
    H * Eps**2 / 2 +   # Internal hardening energy
    sp.Rational(1, 10) * T * eps**2  # Simple thermo-mechanical coupling
)

print("Initial Helmholtz Free Energy F(T, ε, Ɛ):")
display(Math(f"F = {sp.latex(F_expr)}"))
print(f"\nExpression: {F_expr}")

Initial Helmholtz Free Energy F(T, ε, Ɛ):


<IPython.core.display.Math object>


Expression: E*epsilon**2/2 + Epsilon**2*H/2 - T**2*c_T/2 + T*epsilon**2/10


## Create State Function Instance and Thermodynamic Box

Now we create a GSMStateFn instance for the Helmholtz free energy and initialize the GSMThermodynBox2.

In [ ]:
if gsm_state_fn_available and gsm_thermodyn_box2_available:
    # Create the initial state function instance (Helmholtz free energy)
    F_state_fn = GSMStateFn(
        fn_expr=F_expr,
        th_x_var=T,      # Temperature is natural variable
        th_y_var=S,      # Entropy is conjugate variable
        mc_x_var=eps,    # Strain is natural variable
        mc_y_var=sig,    # Stress is conjugate variable
        Eps_var=Eps,     # Internal strain (always extensive)
        Sig_var=Sig,     # Internal stress (conjugate)
        state_function_type=StateFunction.HELMHOLTZ
    )
    
    print("✓ GSMStateFn instance created for Helmholtz free energy")
    print(f"Natural variables: {F_state_fn.get_natural_variables()}")
    print(f"Conjugate variables: {F_state_fn.get_conjugate_variables()}")
    
    # Create the thermodynamic box
    gsm_box = GSMThermodynBox(
        initial_state_fn=StateFunction.HELMHOLTZ,
        initial_state_instance=F_state_fn,
        T_var=T,
        S_var=S
    )
    
    print("\n✓ GSMThermodynBox2 created successfully")
    gsm_box.print_overview()
    
else:
    print("Cannot create thermodynamic box - required modules not available")

✓ GSMStateFn instance created for Helmholtz free energy
Natural variables: [T, epsilon, Epsilon]
Conjugate variables: [S, sigma, Sigma]

✓ GSMThermodynBox2 created successfully
GSM Thermodynamic Box 2 - Overview
Current state function: F
Available state functions: ['F']

Current State Function Details:
GSM State Function Overview
Type: F
Expression: E*epsilon**2/2 + Epsilon**2*H/2 - T**2*c_T/2 + T*epsilon**2/10

Natural Variables (independent):
  Thermal: T
  Mechanical: epsilon
  Internal: Epsilon

Conjugate Variables (derivatives):
  Thermal: S
  Mechanical: sigma
  Internal: Sigma

Expected Variable Organization:
  Natural: ['T', 'eps', 'Eps']
  Conjugate: ['S', 'sig', 'Sig']
  Thermally Intensive: True
  Mechanically Intensive: False
  Transformation Targets: ['U', 'G', 'H']



## Compute All State Functions via Legendre Transformations

Let's compute all four fundamental state functions by performing Legendre transformations.

In [8]:
if 'gsm_box' in locals():
    # Compute all state functions
    all_state_functions = gsm_box.compute_all_state_functions()
    
    print("All State Functions Computed:")
    print("=" * 40)
    
    for state_fn, instance in all_state_functions.items():
        print(f"\n{state_fn.value} - {state_fn.name}:")
        display(Math(f"{state_fn.value} = {sp.latex(instance.fn_expr)}"))
        print(f"Natural vars: {instance.get_natural_variables()}")
        print(f"Conjugate vars: {instance.get_conjugate_variables()}")

All State Functions Computed:

F - HELMHOLTZ:


<IPython.core.display.Math object>

Natural vars: [T, epsilon, Epsilon]
Conjugate vars: [S, sigma, Sigma]

U - INTERNAL_ENERGY:


<IPython.core.display.Math object>

Natural vars: [S, epsilon, Epsilon]
Conjugate vars: [T, sigma, Sigma]

H - ENTHALPY:


<IPython.core.display.Math object>

Natural vars: [S, sigma, Epsilon]
Conjugate vars: [T, epsilon, Sigma]

G - GIBBS:


<IPython.core.display.Math object>

Natural vars: [T, sigma, Epsilon]
Conjugate vars: [S, epsilon, Sigma]


## Create and Display Interactive Widget

Now let's create the interactive widget to explore the thermodynamic relationships.

In [9]:
if gsm_widget_available and 'gsm_box' in locals():
    # Create the interactive widget
    widget = create_interactive_widget(
        gsm_box=gsm_box,
        title="GSM Thermodynamic Box 2 - Interface-based Interactive Widget"
    )
    
    print("✓ Interactive widget created successfully")
    print("\nDisplaying interactive thermodynamic square:")
    print("- Click any button to explore mathematical expressions")
    print("- Blue buttons: Variables (ε, T, -S, -σ)")
    print("- Pink buttons: State functions (F, G, U, H)")
    print("- Darker pink: Initial state function (F)")
    print("- Green button: Information about the thermodynamic square")
    
    # Display the widget
    widget.show()
    
else:
    print("Cannot create widget - required modules not available")

✓ Interactive widget created successfully

Displaying interactive thermodynamic square:
- Click any button to explore mathematical expressions
- Blue buttons: Variables (ε, T, -S, -σ)
- Pink buttons: State functions (F, G, U, H)
- Darker pink: Initial state function (F)
- Green button: Information about the thermodynamic square


## Test Thermodynamic Consistency

Let's verify that our thermodynamic box maintains consistency.

In [10]:
if 'gsm_box' in locals():
    print("Thermodynamic Consistency Check:")
    print("=" * 35)
    
    # Validate consistency
    is_consistent = gsm_box.validate_thermodynamic_consistency()
    print(f"Consistency validation: {'✓ PASS' if is_consistent else '✗ FAIL'}")
    
    # Check available transformations
    transformation_graph = gsm_box.get_transformation_graph()
    print("\nTransformation Graph:")
    for from_state, to_states in transformation_graph.items():
        if to_states:  # Only show states that have transformations
            to_list = [state.value for state in to_states]
            print(f"  {from_state.value} → {', '.join(to_list)}")
    
    # Show expressions summary
    if 'widget' in locals():
        summary = widget.get_expressions_summary()
        print("\nExpressions Summary:")
        for key, value in summary.items():
            if key != 'current_state':
                print(f"  {key}: {value[:50]}{'...' if len(str(value)) > 50 else ''}")
        print(f"  Current state: {summary['current_state']}")

Thermodynamic Consistency Check:
Validation: 4 state functions available
Consistency validation: ✓ PASS

Transformation Graph:
  U → F, H, G
  F → U, H, G
  H → U, F, G
  G → U, F, H

Expressions Summary:
  F: E*epsilon**2/2 + Epsilon**2*H/2 - T**2*c_T/2 + T*e...
  G: E*epsilon**2/2 + Epsilon**2*H/2 - T**2*c_T/2 + T*e...
  U: E*epsilon**2/2 + Epsilon**2*H/2 + S*T - T**2*c_T/2...
  H: E*epsilon**2/2 + Epsilon**2*H/2 + S*T - T**2*c_T/2...
  Current state: F


## Explore State Function Details

Let's examine the details of each state function and their constitutive relations.

In [11]:
if 'gsm_box' in locals():
    print("State Function Constitutive Relations:")
    print("=" * 42)
    
    for state_fn_type in [StateFunction.HELMHOLTZ, StateFunction.GIBBS, 
                          StateFunction.INTERNAL_ENERGY, StateFunction.ENTHALPY]:
        instance = gsm_box.get_state_function(state_fn_type)
        if instance is not None:
            print(f"\n{state_fn_type.value} ({state_fn_type.name}):")
            print("-" * 30)
            
            # Get constitutive relations
            try:
                const_relations = instance.get_constitutive_relations()
                for relation_name, relation_expr in const_relations.items():
                    print(f"  {relation_name}: {relation_expr}")
            except Exception as e:
                print(f"  Error computing constitutive relations: {e}")

State Function Constitutive Relations:

F (HELMHOLTZ):
------------------------------
  Error computing constitutive relations: 'GSMStateFn' object has no attribute 'get_constitutive_relations'

G (GIBBS):
------------------------------
  Error computing constitutive relations: 'GSMStateFn' object has no attribute 'get_constitutive_relations'

U (INTERNAL_ENERGY):
------------------------------
  Error computing constitutive relations: 'GSMStateFn' object has no attribute 'get_constitutive_relations'

H (ENTHALPY):
------------------------------
  Error computing constitutive relations: 'GSMStateFn' object has no attribute 'get_constitutive_relations'


## Test Different Starting Points

Let's create another thermodynamic box starting from a different state function to show the flexibility of the approach.

In [ ]:
if gsm_state_fn_available and gsm_thermodyn_box2_available:
    # Create an internal energy expression U(S, ε, Ɛ)
    U_expr = (
        c_T * S**2 / (2 * T) +  # Thermal contribution (in terms of entropy)
        E * eps**2 / 2 +        # Mechanical elastic energy
        H * Eps**2 / 2          # Internal hardening energy
    )
    
    print("Alternative Starting Point - Internal Energy U(S, ε, Ɛ):")
    display(Math(f"U = {sp.latex(U_expr)}"))
    
    # Create state function instance
    U_state_fn = GSMStateFn(
        fn_expr=U_expr,
        th_x_var=S,      # Entropy is natural variable
        th_y_var=T,      # Temperature is conjugate variable
        mc_x_var=eps,    # Strain is natural variable
        mc_y_var=sig,    # Stress is conjugate variable
        Eps_var=Eps,     # Internal strain (always extensive)
        Sig_var=Sig,     # Internal stress (conjugate)
        state_function_type=StateFunction.INTERNAL_ENERGY
    )
    
    # Create alternative thermodynamic box
    gsm_box_alt = GSMThermodynBox(
        initial_state_fn=StateFunction.INTERNAL_ENERGY,
        initial_state_instance=U_state_fn,
        T_var=T,
        S_var=S
    )
    
    print("\n✓ Alternative GSMThermodynBox2 created (starting from U)")
    
    # Create widget for the alternative box
    if gsm_widget_available:
        widget_alt = create_interactive_widget(
            gsm_box=gsm_box_alt,
            title="Alternative Box - Starting from Internal Energy U"
        )
        
        print("\nAlternative Interactive Widget:")
        widget_alt.show()
    
else:
    print("Cannot create alternative box - required modules not available")

Alternative Starting Point - Internal Energy U(S, ε, Ɛ):


<IPython.core.display.Math object>


✓ Alternative GSMThermodynBox2 created (starting from U)

Alternative Interactive Widget:


## Summary and Conclusions

This notebook demonstrates the interface-based GSM Thermodynamic Box implementation with several key features:

1. **Interface-based Architecture**: Uses GSMStateFnIfc for modular state function implementations
2. **Interactive Visualization**: 3×3 grid widget for exploring thermodynamic relationships
3. **Flexible Starting Points**: Can begin with any of the four fundamental state functions
4. **Automatic Transformations**: Computes missing state functions via Legendre transformations
5. **Educational Value**: Clear visualization of variable relationships and mathematical expressions

The widget provides an intuitive way to explore the thermodynamic square and understand the relationships between different state functions and their natural variables.